# QC — distributions

`participants_qc.csv`, one row per (subject, task), for every cohort that has
one. Seven columns:

| column | catches |
|---|---|
| `mean_fd` | head movement |
| `fd_source` | where FD came from (provenance, not a measure) |
| `peak_isc` | agreement with others watching the same film |
| `best_lag_tr` | wrong stimulus timing |
| `frac_stimulus_covered` | scan stopped early |
| `frac_good_frames` | scrubbing survival |
| `frac_parcels_empty` | registration / coverage failure |

**No threshold is applied here.** These are measurements. Deciding that
`mean_fd > 0.5` is an exclusion belongs with the model that rests on it.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import nbtools as nb

pd.set_option("display.width", 160)

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except ImportError:
    HAVE_MPL = False
    print("matplotlib absent -- tables only")

METRICS = ["mean_fd", "peak_isc", "best_lag_tr", "frac_stimulus_covered",
           "frac_good_frames", "frac_parcels_empty"]

ROOT = nb.output_root()
print("output_root:", ROOT)

In [ ]:
# Every cohort that has a QC table, straight off the meta tree.
paths = sorted((ROOT / "meta" / "cohorts").glob("cohort=*/participants_qc.csv"))
qc = pd.concat([pd.read_csv(p, dtype={"sub": str}) for p in paths], ignore_index=True)
print(f"{len(paths)} cohort(s), {len(qc)} row(s)")
qc.groupby("cohort").agg(rows=("sub", "size"), subs=("sub", "nunique"),
                         tasks=("task", "nunique"))

## Distributions

In [ ]:
# One row per metric, one column per cohort. Count first -- a metric that is
# NaN for a whole cohort is a missing input, not a flat distribution.
summary = (qc.groupby("cohort")[METRICS]
             .agg(["count", "median", "mean", "std", "min", "max"])
             .T.round(3))
summary

In [ ]:
if HAVE_MPL:
    cohorts = sorted(qc["cohort"].unique())
    fig, axes = plt.subplots(len(METRICS), 1, figsize=(7, 2.0 * len(METRICS)))
    for ax, metric in zip(np.atleast_1d(axes), METRICS):
        data = [qc.loc[qc["cohort"] == c, metric].dropna() for c in cohorts]
        keep = [(c, d) for c, d in zip(cohorts, data) if len(d)]
        if keep:
            # Default (vertical) orientation and hand-set tick labels: the
            # kwargs for both were renamed between matplotlib versions, and
            # this spelling works on every one of them.
            ax.boxplot([d for _, d in keep], widths=0.6)
            ax.set_xticks(range(1, len(keep) + 1))
            ax.set_xticklabels([c for c, _ in keep], fontsize=8)
        ax.set_ylabel(metric, fontsize=9)
    fig.tight_layout()

In [ ]:
if HAVE_MPL:
    cohorts = sorted(qc["cohort"].unique())
    fig, axes = plt.subplots(len(METRICS), len(cohorts),
                             figsize=(3.1 * len(cohorts), 1.9 * len(METRICS)))
    axes = np.atleast_2d(axes)
    for i, metric in enumerate(METRICS):
        for j, cohort in enumerate(cohorts):
            ax = axes[i, j]
            values = qc.loc[qc["cohort"] == cohort, metric].dropna()
            if len(values):
                ax.hist(values, bins=25)
            else:
                ax.text(0.5, 0.5, "no data", ha="center", va="center",
                        transform=ax.transAxes, fontsize=8)
            if i == 0:
                ax.set_title(cohort, fontsize=9)
            if j == 0:
                ax.set_ylabel(metric, fontsize=8)
            ax.tick_params(labelsize=7)
    fig.tight_layout()

## `fd_source` — provenance, not a measure

In [ ]:
# Which column FD was read from. A cohort split across two sources is worth
# knowing about before comparing mean_fd across it.
pd.crosstab(qc["cohort"], qc["fd_source"].fillna("(none)"))

## Worst subjects per metric

In [ ]:
# The tail of each distribution, named. Nothing is excluded here -- this is the
# list to look at before deciding what a threshold would cost.
WORST = {"mean_fd": "max", "peak_isc": "min", "frac_stimulus_covered": "min",
         "frac_good_frames": "min", "frac_parcels_empty": "max"}

for metric, direction in WORST.items():
    values = qc[["cohort", "sub", "task", metric]].dropna(subset=[metric])
    if values.empty:
        print(f"\n{metric}: no data"); continue
    tail = (values.nlargest(5, metric) if direction == "max"
            else values.nsmallest(5, metric))
    print(f"\n{metric} ({direction}):")
    print(tail.to_string(index=False))

In [ ]:
# best_lag_tr is two-tailed -- a large lag either way means the stimulus timing
# is wrong, so rank by absolute value.
lags = qc[["cohort", "sub", "task", "best_lag_tr"]].dropna()
if len(lags):
    print(lags.reindex(lags["best_lag_tr"].abs().sort_values(ascending=False).index)
              .head(8).to_string(index=False))
    print("\nlag distribution:",
          dict(lags["best_lag_tr"].round().astype(int).value_counts().sort_index()))
else:
    print("no best_lag_tr values -- ISC needs >= 3 subjects per task")

## If you want to threshold

Nothing above applies one. When you do, do it here rather than in the pipeline,
so a sensitivity analysis can move it:

```python
keep = qc[(qc["mean_fd"] < 0.5) & (qc["frac_good_frames"] > 0.5)]
```

and count what it costs per cohort before committing to it — censoring removes
the most data from the participants a frailty or ageing study is about.

In [ ]:
# What a candidate rule would cost, per cohort. Change the numbers; the point
# is to see the cost before adopting it.
rule = (qc["mean_fd"] < 0.5) & (qc["frac_good_frames"] > 0.5)
pd.DataFrame({
    "rows": qc.groupby("cohort").size(),
    "kept": qc[rule].groupby("cohort").size(),
}).fillna(0).astype(int).assign(
    dropped=lambda d: d["rows"] - d["kept"],
    pct_dropped=lambda d: (100 * (d["rows"] - d["kept"]) / d["rows"]).round(1))